1: Install, seeds, data

In [ ]:
!pip install -q pytorch-crf
!pip install -q gensim || echo "gensim failed - the fallback GloVe loader in Cell 2 will be used"

In [ ]:
import urllib.request, os, random, zipfile
import numpy as np, torch, torch.nn as nn
from collections import Counter
from torchcrf import CRF
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)          # should print: cuda

BASE = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"
os.makedirs("data", exist_ok=True)
for split in ["train", "dev", "test"]:
    fn = f"en_ewt-ud-{split}.conllu"
    if not os.path.exists(f"data/{fn}"):
        urllib.request.urlretrieve(BASE + fn, f"data/{fn}")

def read_conllu(path):
    sents, words, tags = [], [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#"): continue
            if not line:
                if words: sents.append((words, tags)); words, tags = [], []
                continue
            cols = line.split("\t")
            if "-" in cols[0] or "." in cols[0]: continue
            words.append(cols[1]); tags.append(cols[3])
    if words: sents.append((words, tags))
    return sents

train = read_conllu("data/en_ewt-ud-train.conllu")
dev   = read_conllu("data/en_ewt-ud-dev.conllu")
test  = read_conllu("data/en_ewt-ud-test.conllu")
tag_names = sorted({t for _, ts in train for t in ts})
t2i = {t: i for i, t in enumerate(tag_names)}
print(len(train), len(dev), len(test), len(tag_names))

device: cpu
12544 2001 2077 17


2: GloVe and vocabularies

In [13]:
def load_glove():
    """GloVe 6B 100d as dict word -> vector. Tries gensim first, then the Stanford zip."""
    try:
        import gensim.downloader as api
        g = api.load("glove-wiki-gigaword-100")
        return {w: g[w] for w in g.key_to_index}
    except Exception as e:
        print("gensim route failed:", e, "\nDownloading glove.6B.zip (~820 MB, takes a few minutes)...")
        urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", "glove.6B.zip")
        d = {}
        with zipfile.ZipFile("glove.6B.zip") as z, z.open("glove.6B.100d.txt") as f:
            for line in f:
                parts = line.decode("utf-8").rstrip().split(" ")
                d[parts[0]] = np.asarray(parts[1:], dtype="float32")
        return d

glove = load_glove()

word_counts = Counter(w.lower() for ws, _ in train for w in ws)
# vocab: training words + dev/test words that exist in GloVe (no labels used; state this in the report)
extra = {w.lower() for d in (dev, test) for ws, _ in d for w in ws if w.lower() in glove}
words = ["<pad>", "<unk>"] + sorted(set(word_counts) | extra)
w2i = {w: i for i, w in enumerate(words)}

chars = ["<pad>", "<unk>"] + sorted({c for ws, _ in train for w in ws for c in w})
c2i = {c: i for i, c in enumerate(chars)}

emb = np.random.uniform(-0.1, 0.1, (len(words), 100)).astype("float32")
emb[0] = 0
hit = 0
for w, i in w2i.items():
    if w in glove:
        emb[i] = glove[w]; hit += 1
emb = torch.tensor(emb)
print(f"vocab {len(words)}, found in GloVe: {hit}, chars: {len(chars)}")

vocab 18643, found in GloVe: 17048, chars: 110


3: The model

In [14]:
class BiLSTMEncoder(nn.Module):
    """char-BiLSTM (25/dir) + GloVe(100) -> word BiLSTM (100/dir). Reused in Task 3."""
    def __init__(self, emb_matrix, n_chars, char_dim=25, char_hid=25, word_hid=100, dropout=0.5):
        super().__init__()
        self.word_emb = nn.Embedding.from_pretrained(emb_matrix.clone(), freeze=False, padding_idx=0)
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.char_lstm = nn.LSTM(char_dim, char_hid, batch_first=True, bidirectional=True)
        self.word_lstm = nn.LSTM(emb_matrix.size(1) + 2 * char_hid, word_hid,
                                 batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.out_dim = 2 * word_hid

    def forward(self, W, C, lens):
        B, T, L = C.shape
        cl = (C != 0).sum(-1).clamp(min=1).view(-1)                 # true length of each word
        ce = self.drop(self.char_emb(C.view(B * T, L)))
        packed = pack_padded_sequence(ce, cl.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.char_lstm(packed)
        cvec = torch.cat([h[0], h[1]], -1).view(B, T, -1)           # word vector from characters
        x = self.drop(torch.cat([self.word_emb(W), cvec], -1))      # [GloVe ; char]
        packed = pack_padded_sequence(x, lens.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.word_lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=T)
        return self.drop(out)

class BiLSTMCRF(nn.Module):
    def __init__(self, enc, n_tags):
        super().__init__()
        self.enc = enc
        self.fc = nn.Linear(enc.out_dim, n_tags)
        self.crf = CRF(n_tags, batch_first=True)

    def emissions(self, W, C, lens):
        return self.fc(self.enc(W, C, lens))

    def loss(self, W, C, Y, lens, mask):
        return -self.crf(self.emissions(W, C, lens), Y, mask=mask, reduction="mean")

    def decode(self, W, C, lens, mask):
        return self.crf.decode(self.emissions(W, C, lens), mask=mask)

4: Batching and evaluation helpers

In [15]:
MAXC = 20   # truncate very long words to 20 characters

def collate(batch, train_mode=False):
    B = len(batch); T = max(len(ws) for ws, _ in batch)
    L = max(min(len(w), MAXC) for ws, _ in batch for w in ws)
    W = torch.zeros(B, T, dtype=torch.long); C = torch.zeros(B, T, L, dtype=torch.long)
    Y = torch.zeros(B, T, dtype=torch.long); lens = torch.tensor([len(ws) for ws, _ in batch])
    for b, (ws, ts) in enumerate(batch):
        for t, (w, tag) in enumerate(zip(ws, ts)):
            lw = w.lower(); wi = w2i.get(lw, 1)
            if train_mode and word_counts[lw] == 1 and random.random() < 0.5:
                wi = 1                      # singleton -> <unk> half the time, teaches the model to handle unknowns
            W[b, t] = wi; Y[b, t] = t2i[tag]
            for k, ch in enumerate(w[:MAXC]):
                C[b, t, k] = c2i.get(ch, 1)
    mask = torch.arange(T)[None, :] < lens[:, None]
    return W.to(device), C.to(device), Y.to(device), lens, mask.to(device)

def predict(model, data, bs=64):
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(data), bs):
            W, C, Y, lens, mask = collate(data[i:i + bs])
            for p in model.decode(W, C, lens, mask):
                preds.append([tag_names[j] for j in p])
    return preds

def accuracy(model, data):
    preds = predict(model, data)
    c = sum(p == g for pr, (_, gs) in zip(preds, data) for p, g in zip(pr, gs))
    return c / sum(len(gs) for _, gs in data)

5: Training (the long one)

In [16]:
import os

SAVE_DIR = "/content/drive/MyDrive/nlp_assignment"
os.makedirs(SAVE_DIR, exist_ok=True)

best, bad = 0.0, 0

In [22]:
model = BiLSTMCRF(BiLSTMEncoder(emb, len(chars)), len(tag_names)).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.015, momentum=0.9)
BS, MAX_EPOCHS, PATIENCE, LR0 = 16, 50, 5, 0.015
best, bad = 0.0, 0
import os

SAVE_DIR = "/content/drive/MyDrive/nlp_assignment"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(MAX_EPOCHS):

    for g in opt.param_groups:
        g["lr"] = LR0 / (1 + 0.05 * epoch)

    model.train()
    random.shuffle(train)
    total = 0

    for i in range(0, len(train), BS):
        W, C, Y, lens, mask = collate(
            train[i:i + BS],
            train_mode=True
        )

        opt.zero_grad()

        loss = model.loss(W, C, Y, lens, mask)
        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), 5.0
        )

        opt.step()
        total += loss.item()

    dev_acc = accuracy(model, dev)

    print(
        f"epoch {epoch + 1}: "
        f"loss {total / max(1, (len(train) + BS - 1) // BS):.3f} "
        f"| dev acc {dev_acc:.4f}"
    )

    if dev_acc > best:
        best, bad = dev_acc, 0

        torch.save(
            model.state_dict(),
            os.path.join(SAVE_DIR, "bilstm_crf.pt")
        )

        print("Best model saved successfully!")

    else:
        bad += 1

        if bad >= PATIENCE:
            print("Early stopping")
            break

epoch 1: loss 10.819 | dev acc 0.9031
Best model saved successfully!
epoch 2: loss 5.719 | dev acc 0.9246
Best model saved successfully!
epoch 3: loss 4.608 | dev acc 0.9334
Best model saved successfully!
epoch 4: loss 4.071 | dev acc 0.9377
Best model saved successfully!
epoch 5: loss 3.702 | dev acc 0.9411
Best model saved successfully!
epoch 6: loss 3.430 | dev acc 0.9450
Best model saved successfully!
epoch 7: loss 3.252 | dev acc 0.9468
Best model saved successfully!
epoch 8: loss 3.076 | dev acc 0.9470
Best model saved successfully!
epoch 9: loss 2.968 | dev acc 0.9481
Best model saved successfully!
epoch 10: loss 2.854 | dev acc 0.9498
Best model saved successfully!
epoch 11: loss 2.767 | dev acc 0.9497
epoch 12: loss 2.691 | dev acc 0.9519
Best model saved successfully!
epoch 13: loss 2.612 | dev acc 0.9528
Best model saved successfully!
epoch 14: loss 2.552 | dev acc 0.9533
Best model saved successfully!
epoch 15: loss 2.495 | dev acc 0.9536
Best model saved successfully!
epoc

6: Test result (the main number)

In [ ]:
import os
import torch
from collections import Counter

# ---------------------------------
# 1. Load Task 2 trained checkpoint
# ---------------------------------

checkpoint_path = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"Task 2 checkpoint not found: {checkpoint_path}\n"
        "Please complete Task 2 training and save the model first."
    )

model.load_state_dict(
    torch.load(checkpoint_path, map_location=device)
)

model.to(device)
model.eval()

print("Task 2 model loaded successfully!")

# ---------------------------------
# 2. Evaluate model
# ---------------------------------

print("Task 2 dev accuracy :", accuracy(model, dev))
print("Task 2 test accuracy:", accuracy(model, test))

# ---------------------------------
# 3. Known vs Unknown word accuracy
# ---------------------------------

train_vocab = set(word_counts)

preds = predict(model, test)

kc = kn = uc = un = 0
confusions = Counter()
errors = []

for pr, (ws, gs) in zip(preds, test):

    for w, p, g in zip(ws, pr, gs):

        if w.lower() in train_vocab:
            kn += 1
            kc += (p == g)
        else:
            un += 1
            uc += (p == g)

        if p != g:
            confusions[(g, p)] += 1

    if pr != gs and len(errors) < 200:
        errors.append((ws, gs, pr))

# ---------------------------------
# 4. Print accuracy
# ---------------------------------

print("\nKnown vs Unknown Word Accuracy")

print(f"Known words   : {kc}/{kn} = {kc/kn if kn else 0:.4f}")
print(f"Unknown words : {uc}/{un} = {uc/un if un else 0:.4f}")

# ---------------------------------
# 5. Top 10 tag confusions
# ---------------------------------

print("\nTop 10 confusions (gold -> predicted):")

for (g, p), c in confusions.most_common(10):
    print(f"  {g:6s} -> {p:6s} {c}")

# ---------------------------------
# 6. Qualitative error examples
# ---------------------------------

print("\nQualitative Error Examples")

for ws, gs, pr in errors[:3]:

    print()

    for w, g, p in zip(ws, gs, pr):

        marker = "" if g == p else "<-- WRONG"

        print(f"{w:15s} gold={g:6s} pred={p:6s} {marker}")

Task 2 model loaded successfully!
Task 2 dev accuracy : 0.9564577700015906
Task 2 test accuracy: 0.9582768789352036

Known vs Unknown Word Accuracy
Known words   : 22436/23212 = 0.9666
Unknown words : 1611/1882 = 0.8560

Top 10 confusions (gold -> predicted):
  PROPN  -> NOUN   188
  NOUN   -> PROPN  123
  ADJ    -> NOUN   50
  SCONJ  -> ADP    40
  ADV    -> ADP    39
  VERB   -> NOUN   39
  NOUN   -> VERB   39
  NOUN   -> ADJ    38
  ADV    -> ADJ    33
  NOUN   -> NUM    27

Qualitative Error Examples

What            gold=PRON   pred=PRON   
if              gold=SCONJ  pred=SCONJ  
Google          gold=PROPN  pred=PROPN  
expanded        gold=VERB   pred=VERB   
on              gold=ADP    pred=ADP    
its             gold=PRON   pred=PRON   
search          gold=NOUN   pred=NOUN   
-               gold=PUNCT  pred=PUNCT  
engine          gold=NOUN   pred=NOUN   
(               gold=PUNCT  pred=PUNCT  
and             gold=CCONJ  pred=CCONJ  
now             gold=ADV    pred=ADV  

In [ ]:
checkpoint_path = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

print("Checkpoint saved:", os.path.exists(checkpoint_path))
print("Checkpoint size (MB):",
      round(os.path.getsize(checkpoint_path) / (1024 * 1024), 2))

Checkpoint saved: True
Checkpoint size (MB): 7.95


In [21]:
import os
import torch

save_dir = "/content/drive/MyDrive/nlp_assignment"
os.makedirs(save_dir, exist_ok=True)

# Check whether the trained Task 2 model is still in memory
if "model" in globals() and isinstance(model, BiLSTMCRF):

    path = os.path.join(save_dir, "bilstm_crf.pt")

    torch.save(model.state_dict(), path)

    print("SUCCESS: Task 2 checkpoint saved!")
    print("Location:", path)
    print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

else:
    print("Task 2 trained model is not available in memory.")
    print("Do not initialize a new model.")

SUCCESS: Task 2 checkpoint saved!
Location: /content/drive/MyDrive/nlp_assignment/bilstm_crf.pt
Size: 7.95 MB


In [23]:
from google.colab import files
import os

path = "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"

if os.path.exists(path):
    print("Checkpoint found!")
    files.download(path)
else:
    print("Checkpoint not found in this Task 2 Drive.")

Checkpoint found!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
print("Word vocabulary:", len(w2i))
print("Character vocabulary:", len(c2i))
print("Embedding matrix shape:", emb.shape)
print("Checkpoint exists:", os.path.exists(
    "/content/drive/MyDrive/nlp_assignment/bilstm_crf.pt"
))

Word vocabulary: 18643
Character vocabulary: 110
Embedding matrix shape: torch.Size([18643, 100])
Checkpoint exists: True


In [25]:
import os
import torch

# Create the folder if it does not exist
os.makedirs("/content/drive/MyDrive/nlp_assignment", exist_ok=True)

# Save the exact Task 2 vocabulary and embeddings
task2_vocab_bundle = {
    "w2i": w2i,
    "c2i": c2i,
    "emb": emb.cpu()
}

bundle_path = "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"

torch.save(task2_vocab_bundle, bundle_path)

print("Task 2 vocabulary exported successfully!")
print("Word vocabulary:", len(w2i))
print("Character vocabulary:", len(c2i))
print("Embedding shape:", emb.shape)
print("Saved at:", bundle_path)

Task 2 vocabulary exported successfully!
Word vocabulary: 18643
Character vocabulary: 110
Embedding shape: torch.Size([18643, 100])
Saved at: /content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt


In [26]:
from google.colab import files

bundle_path = "/content/drive/MyDrive/nlp_assignment/task2_vocab_bundle.pt"

files.download(bundle_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>